In [1]:
# task 1.1
import sqlite3

con = sqlite3.connect("cu_booking.db")
c = con.cursor()


tables = [
    """
    CREATE TABLE student (
        id TEXT PRIMARY KEY,
        name TEXT,
        contact INTEGER
    )
    """,
    """
    CREATE TABLE cubicle (
        cubicle_no TEXT PRIMARY KEY,
        maintenance INTEGER CHECK (maintenance IN (0,1))
    )
    """,
    """
    CREATE TABLE booking (
        date TEXT,
        cubicle_no TEXT REFERENCES cubicle(cubicle_no),
        student_id TEXT REFERENCES student(id),
        PRIMARY KEY (date,cubicle_no)
    )
    """
]


for table in tables:
    c.execute(table)

con.commit()
con.close()

In [2]:
# task 1.2
import sqlite3
import csv

with open("student.csv") as f:
    student_data = list(csv.reader(f))[1::]

with open("cubicle.csv") as f:
    cubicle_data = list(csv.reader(f))[1::]

with open("booking.csv") as f:
    booking_data = list(csv.reader(f))[1::]

student_query = """
    INSERT INTO student VALUES (?,?,?)
"""
cubicle_query = """
    INSERT INTO cubicle VALUES (?,?)
"""
booking_query = """
    INSERT INTO booking VALUES (?,?,?)
"""

con = sqlite3.connect("cu_booking.db")
c = con.cursor()

c.executemany(student_query,student_data)
c.executemany(cubicle_query,cubicle_data)
c.executemany(booking_query,booking_data)

con.commit()
con.close()

In [3]:
# task 1.3
import sqlite3


query = "UPDATE cubicle SET maintenance = 0 WHERE cubicle_no = ?"

con = sqlite3.connect("cu_booking.db")
c = con.cursor()

c.execute(query,('c01',))

con.commit()
con.close()

In [ ]:
# task 1.4

import sqlite3

query_date = "17112024"

query = "SELECT c.cubicle_no FROM cubicle c LEFT OUTER JOIN booking b ON c.cubicle_no = b.cubicle_no AND b.date = ? WHERE c.maintenance = 0 AND b.cubicle_no is NULL"

con = sqlite3.connect("cu_booking.db")
c = con.cursor()

c.execute(query,(query_date,))
data = list(c.fetchall())

print(data)


con.close()

[('c01',), ('c03',), ('c04',), ('c05',), ('c06',), ('c07',), ('c08',), ('c11',)]


In [9]:
# task 1.5
from flask import Flask, render_template
import sqlite3

app = Flask(__name__)

@app.route('/available/<date>')
def avail(date):
    con = sqlite3.connect("cu_booking.db")
    c = con.cursor()

    cub_query = "SELECT c.cubicle_no FROM cubicle c"

    c.execute(cub_query)
    cubicles = [i[0] for i in c.fetchall()]

    avail_query = "SELECT c.cubicle_no FROM cubicle c LEFT OUTER JOIN booking b ON c.cubicle_no = b.cubicle_no AND b.date = ? WHERE c.maintenance = 0 AND b.cubicle_no is NULL"

    c.execute(avail_query,(date,))
    avail_cub = [i[0] for i in c.fetchall()]

    avails = [(i,"Unavailable") if i not in avail_cub else (i,"Available") for i in cubicles]

    return render_template("index.html",date=date,avails=avails)


app.run()


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [28/Jul/2026 14:53:27] "GET /available/17112024 HTTP/1.1" 200 -
